# 21 · Group points in H3 cells against h3-py

**Question.** Does Fieldwork's `hex_aggregate` widget index points to the right cells, and what
does its sparse-cell rule do to the data?

**This check is weaker than most here, and says so.** Fieldwork indexes with `h3-js` and the
reference with `h3-py` — two bindings of the same C library. The indexing is therefore *not*
independently verified. What is verified is that Fieldwork calls it correctly, orders the result
as it claims, and applies its own sparse-cell rule, which is the part that is Fieldwork's alone.

**Two resolutions, because the example's own setting does not exercise the rule it depends on.**
At resolution 9 all 250 deaths fall into five cells, every one above the minimum, so nothing is
dropped. Resolution 11 produces single-point cells — the case that makes points vanish.

Criteria fixed before running: per resolution, the same cells kept, the same count in each, each
boundary and centre within 1e-9 degrees of h3-py's, the cells ordered by H3 index, and the
matched and dropped point counts exact.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["h3"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_21_hex import run

report = run(Path.cwd() / "fixtures" / "hex-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["points"], "points, minimum occupancy",
      report["minOccupancy"])
print()
print(f"{'resolution':>11} {'cells kept':>11} {'cells dropped':>14} {'points removed':>15} {'worst coord diff':>18}")
for case in report["cases"]:
    worst = max(case["worstBoundaryDifferenceDegrees"], case["worstCentreDifferenceDegrees"])
    print(f"{case['resolution']:>11} {case['cellsKeptReference']:>11} "
          f"{case['cellsDroppedBySparseRule']:>14} {case['pointsDroppedBySparseRule']:>15} "
          f"{worst:>18.1e}")
print()
print(report["environment"]["note"])

## Reading the result

`agrees` means Fieldwork and h3-py put the same points in the same cells with the same
boundaries, and that Fieldwork's own sparse rule was applied as declared. It does not mean H3
indexing is correct — both sides call the same C library — and it does not mean the aggregation
protects anybody.